## Using PulseKit to run PTT pipeline

In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PulseKit import *  
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, r2_score
import os

#### Load Dataset, Get Cardiac Cycles, and process signals

In [2]:
dataset = Dataset(
    datasetName="TestDataset",
    sessionIds=["3", "41"],
    signals=[
        ECG(name="ECG_downsample"),
        PPG(name="PPG_Finger_downsample"),
        PPG(name="PPG_Ear_downsample"),
        PPG(name="PPG_Wrist_downsample"),
        Stethoscope(name="Digital_Stethoscope_downsample"),
        Earbud(name="Earbud_Left_downsample"),
        Earbud(name="Earbud_Right_downsample"),
        Accelerometer(name="AccZ_downsample"),
        RespirationBelt(name="Respiration_Belt_downsample"),
        BP(name="Diastolic_BP_downsample"),
        BP(name="Systolic_BP_downsample"),
        BP(name="NIBP_downsample"),
        Hemodynamics(name="Cardiac_Output_downsample"),
        Hemodynamics(name="Stroke_Volume_downsample"),
        Hemodynamics(name="SVR_downsample"),
        Hemodynamics(name="Heart_Rate_downsample")
    ],
    data_directory="test_data"
)

In [4]:
dataset.get_cardiac_cycles(reference_signal_name="ECG_downsample")

In [ ]:
dataset.process_signals(signal_list=["ECG_downsample", "PPG_Finger_downsample", "PPG_Ear_downsample", "Earbud_Left_downsample", "Systolic_BP_downsample", "Diastolic_BP_downsample"])

Processing session 3...
Processing session 41...


In [ ]:
# Code to process signals, has already been run once shouldnt need to be run again

signals=["ECG", "Digital_Stethoscope", "AccZ", "PPG_Finger", "PPG_Ear", "PPG_Wrist", "Earbud_Left", "Earbud_Right", 'Diastolic_BP', 'Systolic_BP']

#process signals and time how long it takes for each individual signal

for signal in signals:
    print("Processing:", signal)
    dataset.process_signals(signal_list=[signal])

#### Compute features and populate tables with targets

In [6]:
dataset.compute_features(
    features=[
        PWA(dataset=dataset, signals=["PPG_Finger_downsample", "PPG_Ear_downsample"]),
        PAT(dataset=dataset, signals=["PPG_Finger_downsample", "Earbud_Left_downsample", "ECG_downsample"])
    ],
    targets=["Systolic_BP_downsample", "Diastolic_BP_downsample"],
    output_name="test_features"
)

Saved to: test_data/test_features.csv


#### Run Models: e.g. Linear PTT Model for Systolic BP

In [10]:
CALIBRATION_N = 32 
df = pd.read_csv("test_data/test_features.csv")
results = []
np.random.seed(42)


# Keep only relevant columns and drop rows where quality check failed (NaNs)
subset = df[["participant_id", "session_id", "PAT_PPG_Finger_downsample", "Systolic_BP_downsample_midpoint"]].copy()
subset.dropna(inplace=True)
    
# Subject specific modeling
for subject_id in subset["participant_id"].unique():
    subject_data = subset[subset["participant_id"] == subject_id]
        
    X = subject_data["PAT_PPG_Finger_downsample"].values.reshape(-1, 1)
    y = subject_data["Systolic_BP_downsample_midpoint"].values
    
    # Select Calibration Points (Stratified: Low + High values)
    n_half = CALIBRATION_N // 2
    train_idx_low = np.random.choice(np.where(y < np.percentile(y, 50))[0], n_half, replace=False)
    train_idx_high = np.random.choice(np.where(y > np.percentile(y, 50))[0], n_half, replace=False)
    train_idx = np.concatenate([train_idx_low, train_idx_high])
    
    # Data split
    X_train, y_train = X[train_idx], y[train_idx]
    mask = np.ones(len(y), dtype=bool)
    mask[train_idx] = False
    X_test, y_test = X[mask], y[mask]

    # Run model and eval
    model = LinearRegression()
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    
    results.append({
        "subject_id": subject_id,
        "calibration_points": CALIBRATION_N,
        "total_samples": len(subject_data),
        "MAE": round(mae, 3),
        "R2": round(r2, 3),
        "Slope (beta)": round(model.coef_[0], 4),
        "Intercept (alpha)": round(model.intercept_, 4)
    })

final_df = pd.DataFrame(results)
final_df.head()

,subject_id,calibration_points,total_samples,MAE,R2,Slope (beta),Intercept (alpha)
0,3,32,2091,14.469,-0.187,-0.2003,151.9753
1,41,32,1986,11.737,0.095,-0.3040,203.2458
